# MovieLens-100K Core Models (Single Notebook)

This notebook runs the core recommender models implemented under main on MovieLens-100K in one place: data preparation, model training, evaluation, and artifact export.

It includes:
- Session-based baselines from main (Popularity, SKNN variants)
- Neural sequence models from main (GRU4Rec, BERT4Rec, SASRec)
- Embedding-based models from main (LLMSeqSim, LLM2GRU4Rec, LLM2BERT4Rec, LLM2SASRec)

Note: LLM-style models require item embeddings. In this notebook, embeddings are derived from MovieLens genre indicators so everything can run self-contained.

In [ ]:
from __future__ import annotations

import json
import os
import pickle
import subprocess
import sys
import tarfile
import time
import urllib.request
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

REPO_URL = "https://github.com/faroukq1/LLM-Sequential-Recommendation.git"
KAGGLE_REPO_DIR = Path("/kaggle/working/LLM-Sequential-Recommendation")


def is_project_root(path: Path) -> bool:
    return (path / "main").is_dir() and (path / "kaggle").is_dir()


def find_project_root(start: Path) -> Path:
    env_root = os.environ.get("PROJECT_ROOT")
    if env_root:
        candidate = Path(env_root).expanduser().resolve()
        if is_project_root(candidate):
            return candidate

    for candidate in [start, *start.parents]:
        if is_project_root(candidate):
            return candidate

    common_candidates = [
        KAGGLE_REPO_DIR,
        Path("/home/farouk/files/LLM-Sequential-Recommendation"),
    ]
    for candidate in common_candidates:
        if candidate.exists() and is_project_root(candidate):
            return candidate

    if Path("/kaggle/working").exists():
        if not KAGGLE_REPO_DIR.exists():
            print("Project root not found. Cloning repository into /kaggle/working ...")
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(KAGGLE_REPO_DIR)],
                check=True,
            )
        if is_project_root(KAGGLE_REPO_DIR):
            return KAGGLE_REPO_DIR

    raise FileNotFoundError(
        "Could not locate project root containing 'main' and 'kaggle'. "
        "Set PROJECT_ROOT to your repository path.",
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("PROJECT_ROOT:", PROJECT_ROOT)

In [ ]:
from main.data.session_dataset import SessionDataset
from main.data.temporal_split import TemporalSplit
from main.eval.evaluation import Evaluation
from main.eval.metrics.catalog_coverage import CatalogCoverage
from main.eval.metrics.hitrate import HitRate
from main.eval.metrics.metric import MetricDependency
from main.eval.metrics.mrr import MeanReciprocalRank
from main.eval.metrics.ndcg import NormalizedDiscountedCumulativeGain
from main.grurec.grurec import GRURec
from main.grurec.grurec_with_embeddings import GRURecWithEmbeddings
from main.llm_based.similarity_model.llm_seq_sim import LLMSeqSim
from main.popularity.session_popular import SessionBasedPopular
from main.sknn.sknn import SessionBasedCF
from main.transformer.bert.bert import BERT
from main.transformer.bert.bert_with_embeddings import BERTWithEmbeddings
from main.transformer.sasrec.sasrec import SASRec
from main.transformer.sasrec.sasrec_with_embeddings import SASRecWithEmbeddings

DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-100k.zip"

WORK_DIR = PROJECT_ROOT / "kaggle" / "artifacts" / "ml100k_core_models_single_notebook"
DATA_DIR = WORK_DIR / "data"
RESULTS_DIR = WORK_DIR / "results"
RECS_DIR = WORK_DIR / "recommendations"
for directory in [DATA_DIR, RESULTS_DIR, RECS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

INCLUDE_MODELS = [
    "Popular",
    "SKNN",
    "S-SKNN",
    "SF-SKNN",
    "V-SKNN",
    "GRU4Rec",
    "BERT4Rec",
    "SASRec",
    "LLMSeqSim",
    "LLM2GRU4Rec",
    "LLM2BERT4Rec",
    "LLM2SASRec",
]

TOP_KS = [10, 20]

SESSION_GAP_MINUTES = 30
MIN_SESSION_LEN = 2
TEST_FRAC = 0.2
NUM_FOLDS = 0
FILTER_NON_TRAINED_TEST_ITEMS = True

CORES = max(1, min(4, os.cpu_count() or 1))
EVAL_CORES = 1
VERBOSE = True

SKNN_K = 200
SKNN_SAMPLE_SIZE = 1000

MAX_SEQ_LEN = 20
BASE_NEURAL_EMB_DIM = 64
EMBED_NEURAL_EMB_DIM = 20
HIDDEN_DIM = 128
DROP_RATE = 0.2
NUM_EPOCHS = 3
FIT_BATCH_SIZE = 256
PRED_BATCH_SIZE = 1024
TRAIN_VAL_FRACTION = 0.1
EARLY_STOPPING_PATIENCE = 2
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
MASK_PROB = 0.4

print("Work dir:", WORK_DIR)
print("Models:", INCLUDE_MODELS)

In [ ]:
GENRE_COLUMNS = [
    "unknown",
    "Action",
    "Adventure",
    "Animation",
    "Children",
    "Comedy",
    "Crime",
    "Documentary",
    "Drama",
    "Fantasy",
    "FilmNoir",
    "Horror",
    "Musical",
    "Mystery",
    "Romance",
    "SciFi",
    "Thriller",
    "War",
    "Western",
]


def download_movielens_100k(data_dir: Path, dataset_url: str) -> tuple[Path, Path]:
    data_dir.mkdir(parents=True, exist_ok=True)
    udata_path = data_dir / "ml-100k" / "u.data"
    uitem_path = data_dir / "ml-100k" / "u.item"

    if udata_path.exists() and uitem_path.exists():
        return udata_path, uitem_path

    archive_name = dataset_url.rsplit("/", 1)[-1]
    archive_path = data_dir / archive_name

    if not archive_path.exists():
        print("Downloading:", dataset_url)
        urllib.request.urlretrieve(dataset_url, archive_path)

    print("Extracting:", archive_path)
    if archive_path.suffix == ".zip":
        with zipfile.ZipFile(archive_path, "r") as zip_ref:
            zip_ref.extractall(data_dir)
    elif archive_name.endswith(".tar.gz") or archive_name.endswith(".tgz"):
        with tarfile.open(archive_path, "r:gz") as tar_ref:
            tar_ref.extractall(data_dir)
    else:
        raise ValueError(f"Unsupported archive format: {archive_path}")

    if not udata_path.exists() or not uitem_path.exists():
        raise FileNotFoundError("MovieLens-100K files were not found after extraction.")

    return udata_path, uitem_path


def load_movielens_interactions(udata_path: Path) -> pd.DataFrame:
    return pd.read_csv(
        udata_path,
        sep="\t",
        names=["UserId", "ItemId", "Rating", "Timestamp"],
        engine="python",
    )


def sessionize(
    interactions: pd.DataFrame,
    session_gap_minutes: int,
    min_session_len: int,
) -> pd.DataFrame:
    if min_session_len < 2:
        raise ValueError("min_session_len must be >= 2 for next-item evaluation.")

    gap_seconds = session_gap_minutes * 60

    df = interactions.copy()
    df["Timestamp"] = df["Timestamp"].astype(int)
    df["ItemId"] = df["ItemId"].astype(int)
    df["Rating"] = df["Rating"].astype(float)

    df = df.sort_values(["UserId", "Timestamp", "ItemId"]).reset_index(drop=True)

    user_changed = df["UserId"].ne(df["UserId"].shift(1))
    ts_gap = df["Timestamp"].diff().fillna(0)
    new_session = user_changed | (ts_gap > gap_seconds)
    df["SessionId"] = new_session.cumsum().astype(int)

    session_sizes = df.groupby("SessionId").size()
    valid_sessions = session_sizes[session_sizes >= min_session_len].index
    df = df[df["SessionId"].isin(valid_sessions)].copy()

    new_ids = {old_id: idx + 1 for idx, old_id in enumerate(sorted(df["SessionId"].unique()))}
    df["SessionId"] = df["SessionId"].map(new_ids).astype(int)

    df["Time"] = pd.to_datetime(df["Timestamp"], unit="s", utc=True).dt.tz_localize(None)
    df["Time"] = df["Time"].dt.strftime("%Y-%m-%d %H:%M:%S.%f")
    df["Reward"] = df["Rating"].astype(float)

    out_df = df[["SessionId", "ItemId", "Time", "Reward"]]
    out_df = out_df.sort_values(["SessionId", "Time", "ItemId"]).reset_index(drop=True)
    return out_df


def load_movielens_item_metadata(uitem_path: Path) -> pd.DataFrame:
    columns = [
        "ItemId",
        "Title",
        "ReleaseDate",
        "VideoReleaseDate",
        "IMDbURL",
        *GENRE_COLUMNS,
    ]
    return pd.read_csv(
        uitem_path,
        sep="|",
        names=columns,
        usecols=list(range(len(columns))),
        encoding="latin-1",
        engine="python",
    )


def build_genre_embedding_tables(
    item_metadata: pd.DataFrame,
    item_ids: np.ndarray,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    filtered = item_metadata[item_metadata["ItemId"].isin(item_ids)].copy()
    if filtered.empty:
        raise ValueError("No overlapping item ids found for item metadata.")

    genre_matrix = filtered[GENRE_COLUMNS].to_numpy(dtype=np.float32)

    no_genre = (genre_matrix.sum(axis=1, keepdims=True) == 0).astype(np.float32)
    embeddings = np.concatenate([genre_matrix, no_genre], axis=1)

    norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
    norms[norms == 0] = 1.0
    embeddings = embeddings / norms

    item_data = filtered[["ItemId"]].copy()
    item_data["embedding"] = [row.astype(np.float32) for row in embeddings]
    item_data["class"] = np.argmax(embeddings, axis=1).astype(int)
    item_data["category_size"] = [[] for _ in range(len(item_data))]

    embeddings_csv_df = item_data[["ItemId", "embedding", "class"]].copy()
    embeddings_csv_df["embedding"] = embeddings_csv_df["embedding"].apply(
        lambda vec: json.dumps([float(x) for x in vec.tolist()])
    )

    return item_data, embeddings_csv_df


def build_dataset(
    sessions_csv_path: Path,
    dataset_pickle_path: Path,
) -> SessionDataset:
    dataset = SessionDataset(str(sessions_csv_path), n_withheld=1, evolving=False)
    split_strategy = TemporalSplit(
        test_frac=TEST_FRAC,
        num_folds=NUM_FOLDS,
        filter_non_trained_test_items=FILTER_NON_TRAINED_TEST_ITEMS,
        fold_strategy="chain",
    )
    dataset.load_and_split(split_strategy=split_strategy)
    dataset.to_pickle(str(dataset_pickle_path))
    return dataset

In [ ]:
udata_path, uitem_path = download_movielens_100k(DATA_DIR, DATASET_URL)
interactions = load_movielens_interactions(udata_path)
sessions_df = sessionize(
    interactions=interactions,
    session_gap_minutes=SESSION_GAP_MINUTES,
    min_session_len=MIN_SESSION_LEN,
)

sessions_csv_path = DATA_DIR / "sessions_ml100k.csv"
sessions_df.to_csv(sessions_csv_path, index=False)

dataset_pickle_path = DATA_DIR / "dataset_ml100k.pickle"
dataset = build_dataset(sessions_csv_path, dataset_pickle_path)

item_metadata = load_movielens_item_metadata(uitem_path)
item_ids = np.sort(dataset.get_input_data()["ItemId"].unique())
item_data, embeddings_csv_df = build_genre_embedding_tables(item_metadata, item_ids)
dataset.set_item_data(item_data)

embeddings_csv_path = DATA_DIR / "ml100k_item_embeddings.csv"
embeddings_csv_df.to_csv(embeddings_csv_path, index=False)

print("Sessions CSV:", sessions_csv_path)
print("Dataset pickle:", dataset_pickle_path)
print("Embeddings CSV:", embeddings_csv_path)
print("#Interactions:", len(dataset.get_input_data()))
print("#Sessions:", dataset.get_unique_sample_count())
print("#Unique items:", dataset.get_unique_item_count())

In [ ]:
def instantiate_models(embedding_csv_path: Path) -> dict[str, Any]:
    shared = {
        "cores": CORES,
        "is_verbose": VERBOSE,
    }
    neural_common = {
        "cores": CORES,
        "is_verbose": VERBOSE,
        "num_epochs": NUM_EPOCHS,
        "fit_batch_size": FIT_BATCH_SIZE,
        "pred_batch_size": PRED_BATCH_SIZE,
        "train_val_fraction": TRAIN_VAL_FRACTION,
        "early_stopping_patience": EARLY_STOPPING_PATIENCE,
        "pred_seen": False,
    }
    optimizer_kwargs = {
        "learning_rate": LEARNING_RATE,
        "weight_decay": WEIGHT_DECAY,
    }

    models: dict[str, Any] = {
        "Popular": SessionBasedPopular(**shared),
        "SKNN": SessionBasedCF(
            k=SKNN_K,
            sample_size=SKNN_SAMPLE_SIZE,
            sampling="recent",
            similarity_measure="cosine",
            idf_weighting=True,
            filter_prompt_items=True,
            **shared,
        ),
        "S-SKNN": SessionBasedCF(
            k=SKNN_K,
            sample_size=SKNN_SAMPLE_SIZE,
            sampling="recent",
            similarity_measure="cosine",
            idf_weighting=True,
            sequential_weighting=True,
            filter_prompt_items=True,
            **shared,
        ),
        "SF-SKNN": SessionBasedCF(
            k=SKNN_K,
            sample_size=SKNN_SAMPLE_SIZE,
            sampling="recent",
            similarity_measure="cosine",
            idf_weighting=True,
            sequential_filter=True,
            filter_prompt_items=True,
            **shared,
        ),
        "V-SKNN": SessionBasedCF(
            k=SKNN_K,
            sample_size=SKNN_SAMPLE_SIZE,
            sampling="recent",
            similarity_measure="dot",
            idf_weighting=True,
            decay="harmonic",
            filter_prompt_items=True,
            **shared,
        ),
        "GRU4Rec": GRURec(
            N=MAX_SEQ_LEN,
            emb_dim=BASE_NEURAL_EMB_DIM,
            hidden_dim=HIDDEN_DIM,
            drop_rate=DROP_RATE,
            activation="relu",
            optimizer_kwargs=optimizer_kwargs,
            **neural_common,
        ),
        "BERT4Rec": BERT(
            N=MAX_SEQ_LEN,
            L=1,
            h=2,
            emb_dim=BASE_NEURAL_EMB_DIM,
            drop_rate=DROP_RATE,
            mask_prob=MASK_PROB,
            activation="gelu",
            optimizer_kwargs=optimizer_kwargs,
            transformer_layer_kwargs={"layout": "FDRN"},
            **neural_common,
        ),
        "SASRec": SASRec(
            N=MAX_SEQ_LEN,
            L=1,
            h=2,
            emb_dim=BASE_NEURAL_EMB_DIM,
            drop_rate=DROP_RATE,
            activation="relu",
            optimizer_kwargs=optimizer_kwargs,
            transformer_layer_kwargs={"layout": "NFDR"},
            **neural_common,
        ),
        "LLMSeqSim": LLMSeqSim(
            cores=CORES,
            is_verbose=VERBOSE,
            similarity_measure="cosine",
            embedding_combination_strategy="last",
            combination_decay=None,
            filter_prompt_items=True,
            batch_size=PRED_BATCH_SIZE,
            dim_reduction_config=None,
        ),
        "LLM2GRU4Rec": GRURecWithEmbeddings(
            product_embeddings_location=str(embedding_csv_path),
            red_method="PCA",
            red_params={},
            N=MAX_SEQ_LEN,
            emb_dim=EMBED_NEURAL_EMB_DIM,
            hidden_dim=HIDDEN_DIM,
            drop_rate=DROP_RATE,
            activation="relu",
            optimizer_kwargs=optimizer_kwargs,
            **neural_common,
        ),
        "LLM2BERT4Rec": BERTWithEmbeddings(
            product_embeddings_location=str(embedding_csv_path),
            red_method="PCA",
            red_params={},
            N=MAX_SEQ_LEN,
            L=1,
            h=2,
            emb_dim=EMBED_NEURAL_EMB_DIM,
            drop_rate=DROP_RATE,
            mask_prob=MASK_PROB,
            activation="gelu",
            optimizer_kwargs=optimizer_kwargs,
            transformer_layer_kwargs={"layout": "FDRN"},
            **neural_common,
        ),
        "LLM2SASRec": SASRecWithEmbeddings(
            product_embeddings_location=str(embedding_csv_path),
            red_method="PCA",
            red_params={},
            N=MAX_SEQ_LEN,
            L=1,
            h=2,
            emb_dim=EMBED_NEURAL_EMB_DIM,
            drop_rate=DROP_RATE,
            activation="relu",
            optimizer_kwargs=optimizer_kwargs,
            transformer_layer_kwargs={"layout": "NFDR"},
            **neural_common,
        ),
    }

    return models


models = instantiate_models(embeddings_csv_path)
print("Available models:", sorted(models.keys()))

In [ ]:
metric_list = [
    NormalizedDiscountedCumulativeGain(),
    HitRate(),
    MeanReciprocalRank(),
    CatalogCoverage(),
]
dependencies = {
    MetricDependency.NUM_ITEMS: dataset.get_unique_item_count(),
}


def train_and_predict(model: Any, dataset: SessionDataset, item_data: pd.DataFrame, top_k: int) -> dict[int, np.ndarray]:
    train_data = dataset.get_train_data()
    test_prompts = dataset.get_test_prompts()

    if isinstance(model, LLMSeqSim):
        model.train(train_data, item_data)
    elif isinstance(model, SessionBasedCF) and model.use_item_embeddings:
        model.train(train_data, item_data)
    else:
        model.train(train_data)

    return model.predict(test_prompts, top_k=top_k)


results_rows: list[dict[str, Any]] = []
failure_rows: list[dict[str, Any]] = []

for requested_name in INCLUDE_MODELS:
    if requested_name not in models:
        failure_rows.append({
            "RequestedModel": requested_name,
            "ErrorType": "UnknownModel",
            "Error": "Model is not present in instantiate_models().",
        })
        print(f"SKIP {requested_name}: not available")
        continue

    model = models[requested_name]
    print(f"\n=== Running {requested_name} ===")

    try:
        start = time.perf_counter()
        recommendations = train_and_predict(model, dataset, item_data, top_k=max(TOP_KS))
        elapsed = time.perf_counter() - start

        recs_path = RECS_DIR / f"recs_{requested_name}.pickle"
        with open(recs_path, "wb") as write_file:
            pickle.dump(recommendations, write_file)

        row: dict[str, Any] = {
            "RequestedModel": requested_name,
            "Model": model.name(),
            "TrainPredictSeconds": round(elapsed, 4),
        }

        for top_k in TOP_KS:
            report = Evaluation.eval(
                predictions=recommendations,
                ground_truths=dataset.get_test_ground_truths(),
                top_k=top_k,
                metrics=metric_list,
                metrics_per_sample=False,
                dependencies=dependencies,
                cores=EVAL_CORES,
                model_name=model.name(),
            )
            for metric_name, value in report.results.items():
                row[metric_name] = float(value)

        results_rows.append(row)
        print(f"OK {requested_name} -> {model.name()} ({elapsed:.2f}s)")
    except Exception as exc:
        failure_rows.append({
            "RequestedModel": requested_name,
            "ErrorType": type(exc).__name__,
            "Error": str(exc),
        })
        print(f"FAILED {requested_name}: {type(exc).__name__}: {exc}")

results_df = pd.DataFrame(results_rows)
if not results_df.empty:
    sort_col = "NDCG@20" if "NDCG@20" in results_df.columns else results_df.columns[-1]
    results_df = results_df.sort_values(sort_col, ascending=False).reset_index(drop=True)

failures_df = pd.DataFrame(failure_rows)

print("\n=== Results Preview ===")
display(results_df)
print("\n=== Failures Preview ===")
display(failures_df)

In [ ]:
results_csv = RESULTS_DIR / "core_model_results_ml100k.csv"
failures_csv = RESULTS_DIR / "core_model_failures_ml100k.csv"

results_df.to_csv(results_csv, index=False)
failures_df.to_csv(failures_csv, index=False)

print("Saved results to:", results_csv)
print("Saved failures to:", failures_csv)
print("Saved recommendation pickles to:", RECS_DIR)

## Notes

- LLM-style models in this notebook use genre-derived embeddings, not external OpenAI/PaLM embeddings.
- You can reduce runtime by lowering NUM_EPOCHS or shrinking INCLUDE_MODELS.
- Results are written under kaggle/artifacts/ml100k_core_models_single_notebook/results.